# Preparación, limpieza y estructuración de datos: churn en plataformas digitales

**Actividad:** GA1-220501094-01-AA1-EV07 · Preparación, limpieza y estructuración de datos del proyecto individual con Pandas

**Programa:** Tecnólogo en Análisis y Desarrollo de Software · CTMA – SENA, Regional Antioquia · Ficha 3223873

**Aprendiz:** Nicolás Muñoz Hernández

**Avance de este cuaderno:** Lecciones 2, 3 y 4 (limpieza, análisis y estructuración)

**Proyecto:** Análisis y predicción de cancelación de suscripciones (churn) en plataformas digitales

**Pregunta orientadora:** ¿cómo preparar y estructurar los datos del proyecto para obtener información confiable que permita responder la pregunta analítica?

**Pregunta analítica del proyecto:** ¿qué patrones de interacción, frecuencia de uso y nivel de soporte técnico en los primeros 30 días permiten predecir el riesgo de cancelación?

> **Alcance ajustado.** El diagnóstico de calidad (`data_quality.ipynb` y `reports/data_quality.md`) concluyó que ningún dataset trae información por ventana de 30 días. Por eso este cuaderno responde con las variables disponibles: uso, inicios de sesión, tickets y llamadas de soporte, y fallos de pago.

| Parte | Contenido |
|---|---|
| Preparación | Carga de los dos datasets y estado inicial |
| Lección 2 | Duplicados, nulos, tipos, estandarización, funciones, pipeline y validaciones |
| Lección 3 | Filtros, ordenamiento, agrupaciones, indicadores y tablas dinámicas |
| Lección 4 | Relación entre datasets, estructuración, validación y exportación a `data/processed/` |

## Preparación: carga de los datos y estado inicial

Los dos archivos originales están en `data/raw/` y **no se modifican**: toda la limpieza se hace sobre copias en memoria y los resultados se guardan en `data/processed/`.

In [34]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# El cuaderno vive en notebooks/, así que los datos están un nivel arriba.
RUTA = next((c for c in [Path('../data/raw'), Path('data/raw')] if (c / 'churn.csv').exists()), None)
assert RUTA is not None, 'No encuentro los CSV: deben estar en data/raw/ (ver README.md).'
RAIZ = RUTA.parent.parent
SALIDA = RAIZ / 'data' / 'processed'
SALIDA.mkdir(parents=True, exist_ok=True)

fmt = lambda n: f'{n:,.0f}'.replace(',', '.')      # 1234567 -> 1.234.567

try:
    import pyarrow
    print('pyarrow', pyarrow.__version__)
except ImportError:
    print('AVISO: falta pyarrow. Instálalo con  python -m pip install pyarrow  y reinicia el kernel.')
print('pandas', pd.__version__)
print('Datos leídos desde:', RUTA.resolve())

pyarrow 26.0.1
pandas 3.1.1
Datos leídos desde: C:\Users\IK\Documents\proyecto-churn\data\raw


In [35]:
sus_raw = pd.read_csv(RUTA / 'customer_subscription_churn_usage_patterns.csv')
telco_raw = pd.read_csv(RUTA / 'churn.csv')

print('Suscripción (SaaS):', sus_raw.shape)
print('Telco:             ', telco_raw.shape)
# Deberías ver (2800, 10) y (4250, 21)

Suscripción (SaaS): (2800, 10)
Telco:              (4250, 21)


### Estado inicial de los datos

Se define una función que resume la calidad de una tabla. Se usará de nuevo al final para comparar el antes y el después.

In [36]:
def perfil_calidad(df, nombre, etapa):
    texto = df.select_dtypes(include=['object', 'string', 'category']).columns
    return {
        'dataset': nombre,
        'etapa': etapa,
        'filas': len(df),
        'columnas': df.shape[1],
        'nulos': int(df.isna().sum().sum()),
        'duplicados': int(df.duplicated().sum()),
        'columnas_texto_o_categoria': len(texto),
        'memoria_kb': round(df.memory_usage(deep=True).sum() / 1e3, 1),
    }

estado = [perfil_calidad(sus_raw, 'suscripción', '1 inicial'),
          perfil_calidad(telco_raw, 'telco', '1 inicial')]
pd.DataFrame(estado)

,dataset,etapa,filas,columnas,nulos,duplicados,columnas_texto_o_categoria,memoria_kb
0,suscripción,1 inicial,2800,10,0,0,3,278.0
1,telco,1 inicial,4250,21,0,0,3,841.6


In [37]:
print(sus_raw.dtypes.astype(str).to_dict())
print()
print(telco_raw.dtypes.astype(str).to_dict())

{'user_id': 'int64', 'signup_date': 'str', 'plan_type': 'str', 'monthly_fee': 'int64', 'avg_weekly_usage_hours': 'float64', 'support_tickets': 'int64', 'payment_failures': 'int64', 'tenure_months': 'int64', 'last_login_days_ago': 'int64', 'churn': 'str'}

{'customer_id': 'str', 'state': 'str', 'account_length': 'int64', 'area_code': 'str', 'international_plan': 'bool', 'voice_mail_plan': 'bool', 'number_vmail_messages': 'int64', 'total_day_minutes': 'float64', 'total_day_calls': 'int64', 'total_day_charge': 'float64', 'total_eve_minutes': 'float64', 'total_eve_calls': 'int64', 'total_eve_charge': 'float64', 'total_night_minutes': 'float64', 'total_night_calls': 'int64', 'total_night_charge': 'float64', 'total_intl_minutes': 'float64', 'total_intl_calls': 'int64', 'total_intl_charge': 'float64', 'number_customer_service_calls': 'int64', 'churn': 'bool'}


---
# Lección 2 · Limpieza y estandarización

Para cada problema hay tres opciones: **eliminar, corregir o conservar**. La decisión depende de qué significa el dato, no solo de que "se vea raro". Orden de trabajo: duplicados → nulos → tipos → texto → comprobaciones → pipeline → validación.

## 2.1 Duplicados

Se revisan los duplicados exactos y también los que se repiten **ignorando el identificador** (dos filas iguales con distinto `user_id` serían el mismo cliente registrado dos veces).

In [38]:
def revisar_duplicados(df, id_col):
    return {
        'duplicados exactos': int(df.duplicated().sum()),
        'identificadores repetidos': int(df[id_col].duplicated().sum()),
        'filas iguales ignorando el identificador': int(df.drop(columns=id_col).duplicated().sum()),
    }

dup = pd.DataFrame({'suscripción': revisar_duplicados(sus_raw, 'user_id'),
                    'telco': revisar_duplicados(telco_raw, 'customer_id')})
dup

,suscripción,telco
duplicados exactos,0,0
identificadores repetidos,0,0
filas iguales ignorando el identificador,0,0


In [39]:
total_dup = int(dup.values.sum())
display(Markdown(
    "**Decisión:** " + (
        "no hay duplicados de ninguno de los tres tipos, así que **no se elimina ninguna fila**. Aun así, la función `quitar_duplicados` se incluye en el pipeline para que la limpieza siga siendo válida si llegan datos nuevos."
        if total_dup == 0 else
        f"hay {total_dup} casos que se deben revisar antes de eliminar."
    )
))

**Decisión:** no hay duplicados de ninguno de los tres tipos, así que **no se elimina ninguna fila**. Aun así, la función `quitar_duplicados` se incluye en el pipeline para que la limpieza siga siendo válida si llegan datos nuevos.

## 2.2 Valores faltantes

Un valor faltante no siempre aparece como `NaN`: también puede venir como texto vacío, como la palabra `NA` o como un cero que no es real. Se revisan las tres formas.

In [40]:
def revisar_faltantes(df):
    nulos = int(df.isna().sum().sum())
    texto = df.select_dtypes(include=['object', 'string'])
    vacios = int(sum(texto[c].str.strip().isin(['', 'NA', 'N/A', 'nan', 'NaN', 'null', 'None']).sum() for c in texto))
    return {'nulos (NaN)': nulos, 'texto vacío o "NA"': vacios}

pd.DataFrame({'suscripción': revisar_faltantes(sus_raw), 'telco': revisar_faltantes(telco_raw)})

,suscripción,telco
nulos (NaN),0,0
"texto vacío o ""NA""",0,0


**Ceros sospechosos.** En el diagnóstico se vio que hay clientes de Telco con 0 minutos diurnos pero con uso normal en las otras franjas. Se revisan con detalle.

In [41]:
cero_dia = telco_raw[telco_raw['total_day_minutes'] == 0]
print('Clientes con 0 minutos diurnos:', len(cero_dia))
display(cero_dia[['customer_id', 'total_day_minutes', 'total_day_calls', 'total_day_charge',
                  'total_eve_minutes', 'total_night_minutes', 'churn']])

# Referencia: los ceros en minutos internacionales (clientes sin plan internacional) son normales
print('Clientes con 0 minutos internacionales:', int((telco_raw['total_intl_minutes'] == 0).sum()))

Clientes con 0 minutos diurnos: 2


,customer_id,total_day_minutes,total_day_calls,total_day_charge,total_eve_minutes,total_night_minutes,churn
341,55058c62-ff1a-4cff-aecd-fd2d2e0096d5,0.0,0,0.0,192.1,168.8,False
1538,308d9375-7391-40be-9958-b7bb59c42a38,0.0,0,0.0,159.6,167.1,True


Clientes con 0 minutos internacionales: 22


**Decisión sobre faltantes**

| Columna | Situación | Decisión |
|---|---|---|
| Todas las columnas | No hay nulos ni textos vacíos | No se imputa nada: rellenar datos que no faltan los alteraría |
| `total_day_minutes` (Telco) | 2 clientes con 0 minutos, 0 llamadas y 0 cargo diurno, pero con uso normal de noche y tarde | **Conservar y marcar** con una columna indicadora (`zero_day_usage`, se crea en la Lección 4). No se imputa ni se borra: son 2 de 4.250 y no se sabe si el cero es real |
| `total_intl_minutes` (Telco) | Los 0 corresponden a clientes sin llamadas internacionales | Conservar: es un valor real |

## 2.3 Tipos de dato

Los tipos se corrigen con `astype`, `to_datetime` y `Categorical`. Se reduce el tamaño de los enteros y se convierte `churn` a `0/1` en **ambos** datasets (en uno era `Yes/No` y en el otro `True/False`).

In [42]:
# Comprobar antes de convertir: ¿los valores caben en los tipos más pequeños y las etiquetas son las esperadas?
print('churn suscripción:', sorted(sus_raw['churn'].unique()))
print('churn telco:      ', sorted(telco_raw['churn'].unique().tolist()))
print('plan_type:        ', sorted(sus_raw['plan_type'].unique()))
print('Máximos suscripción:', sus_raw[['user_id', 'monthly_fee', 'support_tickets', 'payment_failures', 'tenure_months', 'last_login_days_ago']].max().to_dict())
print('Máximos telco:', telco_raw[['account_length', 'number_vmail_messages', 'total_day_calls', 'total_eve_calls', 'total_night_calls', 'total_intl_calls', 'number_customer_service_calls']].max().to_dict())

churn suscripción: ['No', 'Yes']
churn telco:       [False, True]
plan_type:         ['Basic', 'Premium', 'Standard']
Máximos suscripción: {'user_id': 2800, 'monthly_fee': 699, 'support_tickets': 8, 'payment_failures': 5, 'tenure_months': 36, 'last_login_days_ago': 60}
Máximos telco: {'account_length': 243, 'number_vmail_messages': 52, 'total_day_calls': 165, 'total_eve_calls': 170, 'total_night_calls': 175, 'total_intl_calls': 20, 'number_customer_service_calls': 9}


## 2.4 Estandarización de textos y formatos

- `plan_type`: se quitan espacios y se unifican las mayúsculas, y se convierte en categoría **ordenada** (Basic < Standard < Premium).
- `state`: mayúsculas sin espacios.
- `area_code`: llega como `area_code_510`; se deja solo el número (`510`).
- Nombres de columnas: minúsculas y sin espacios.

In [43]:
print(sorted(telco_raw['area_code'].unique()))
print(telco_raw['state'].str.len().value_counts().to_dict())
print((sus_raw['plan_type'] != sus_raw['plan_type'].str.strip().str.title()).sum(), 'planes con texto por corregir')

['area_code_408', 'area_code_415', 'area_code_510']
{2: 4250}
0 planes con texto por corregir


## 2.5 Funciones reutilizables y pipeline de limpieza

Cada paso es una función pequeña que recibe una tabla y devuelve una tabla nueva (sin modificar la original). Se encadenan con `pipe()`, y después de cada paso se anota el estado de la tabla para tener un **registro de cambios**.

In [44]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla, 'paso': paso, 'filas': len(df), 'columnas': df.shape[1],
        'nulos': int(df.isna().sum().sum()), 'memoria_kb': round(df.memory_usage(deep=True).sum() / 1e3, 1),
    })
    return df

def estandarizar_columnas(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    return df

def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def limpiar_texto(df, columnas, caso='title'):
    df = df.copy()
    for c in columnas:
        s = df[c].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
        df[c] = getattr(s.str, caso)()
    return df

def tipos_suscripcion(df):
    df = df.copy()
    df['signup_date'] = pd.to_datetime(df['signup_date'], format='%Y-%m-%d')
    df['plan_type'] = pd.Categorical(df['plan_type'], categories=['Basic', 'Standard', 'Premium'], ordered=True)
    df['churn'] = df['churn'].map({'Yes': 1, 'No': 0})
    return df.astype({'user_id': 'int16', 'monthly_fee': 'int16', 'support_tickets': 'int8',
                      'payment_failures': 'int8', 'tenure_months': 'int8',
                      'last_login_days_ago': 'int8', 'churn': 'int8'})

def tipos_telco(df):
    df = df.copy()
    df['area_code'] = df['area_code'].str.replace('area_code_', '', regex=False)
    return df.astype({'state': 'category', 'area_code': 'category', 'churn': 'int8',
                      'account_length': 'int16', 'number_vmail_messages': 'int16',
                      'total_day_calls': 'int16', 'total_eve_calls': 'int16',
                      'total_night_calls': 'int16', 'total_intl_calls': 'int16',
                      'number_customer_service_calls': 'int8'})

In [45]:
registro = []   # se reinicia aquí para no duplicar filas si se vuelve a ejecutar la celda

sus = (sus_raw
    .pipe(anotar, 'suscripción', '0 original')
    .pipe(estandarizar_columnas)
    .pipe(quitar_duplicados)
    .pipe(anotar, 'suscripción', '1 sin duplicados')
    .pipe(limpiar_texto, ['plan_type', 'churn'], caso='title')
    .pipe(anotar, 'suscripción', '2 texto estandarizado')
    .pipe(tipos_suscripcion)
    .pipe(anotar, 'suscripción', '3 tipos corregidos'))

telco = (telco_raw
    .pipe(anotar, 'telco', '0 original')
    .pipe(estandarizar_columnas)
    .pipe(quitar_duplicados)
    .pipe(anotar, 'telco', '1 sin duplicados')
    .pipe(limpiar_texto, ['state'], caso='upper')
    .pipe(anotar, 'telco', '2 texto estandarizado')
    .pipe(tipos_telco)
    .pipe(anotar, 'telco', '3 tipos corregidos'))

informe_cambios = pd.DataFrame(registro)
informe_cambios

,tabla,paso,filas,columnas,nulos,memoria_kb
0,suscripción,0 original,2800,10,0,278.0
1,suscripción,1 sin duplicados,2800,10,0,278.0
2,suscripción,2 texto estandarizado,2800,10,0,278.7
3,suscripción,3 tipos corregidos,2800,10,0,73.1
4,telco,0 original,4250,21,0,841.6
5,telco,1 sin duplicados,4250,21,0,841.6
6,telco,2 texto estandarizado,4250,21,0,842.2
7,telco,3 tipos corregidos,4250,21,0,536.2


In [46]:
ic = informe_cambios.set_index(['tabla', 'paso'])
ms0, ms3 = ic.loc[('suscripción', '0 original'), 'memoria_kb'], ic.loc[('suscripción', '3 tipos corregidos'), 'memoria_kb']
mt0, mt3 = ic.loc[('telco', '0 original'), 'memoria_kb'], ic.loc[('telco', '3 tipos corregidos'), 'memoria_kb']
display(Markdown(
    f"**Interpretación del registro de cambios:**\n\n"
    f"- Ningún paso cambió el número de filas ni de nulos: no había duplicados ni faltantes que eliminar.\n"
    f"- El único cambio de tamaño viene de los **tipos**: la tabla de suscripción pasó de {ms0:,.0f} a {ms3:,.0f} KB ({(1 - ms3 / ms0) * 100:.0f} % menos) y la de Telco de {mt0:,.0f} a {mt3:,.0f} KB ({(1 - mt3 / mt0) * 100:.0f} % menos)."
))

**Interpretación del registro de cambios:**

- Ningún paso cambió el número de filas ni de nulos: no había duplicados ni faltantes que eliminar.
- El único cambio de tamaño viene de los **tipos**: la tabla de suscripción pasó de 278 a 73 KB (74 % menos) y la de Telco de 842 a 536 KB (36 % menos).

## 2.6 Validación de la limpieza

Las comprobaciones con `assert` detienen el cuaderno si algo no es como se espera. Así la limpieza queda verificada y no solo "a simple vista".

In [47]:
def validar_suscripcion(df):
    assert df['user_id'].is_unique, 'user_id repetido'
    assert df.isna().sum().sum() == 0, 'hay nulos'
    assert df['churn'].isin([0, 1]).all(), 'churn fuera de 0/1'
    assert str(df['signup_date'].dtype).startswith('datetime64'), 'signup_date no es fecha'
    assert df['plan_type'].notna().all(), 'plan_type con valores desconocidos'
    assert df.groupby('plan_type', observed=True)['monthly_fee'].nunique().eq(1).all(), 'una tarifa por plan'
    assert df['avg_weekly_usage_hours'].between(0, 168).all(), 'horas de uso imposibles'
    assert df[['support_tickets', 'payment_failures', 'tenure_months', 'last_login_days_ago']].ge(0).all().all(), 'valores negativos'
    return 'Suscripción: validación superada'

def validar_telco(df):
    assert df['customer_id'].is_unique, 'customer_id repetido'
    assert df.isna().sum().sum() == 0, 'hay nulos'
    assert df['churn'].isin([0, 1]).all(), 'churn fuera de 0/1'
    assert set(df['area_code'].unique()) == {'408', '415', '510'}, 'códigos de área inesperados'
    assert df['state'].str.len().eq(2).all(), 'estado con formato incorrecto'
    minutos = [c for c in df.columns if c.endswith('_minutes')]
    assert df[minutos].ge(0).all().all(), 'minutos negativos'
    assert df['number_customer_service_calls'].between(0, 20).all(), 'llamadas de soporte fuera de rango'
    return 'Telco: validación superada'

print(validar_suscripcion(sus))
print(validar_telco(telco))
print(sus.shape, telco.shape)
# Deberías ver (2800, 10) y (4250, 21), igual que al inicio: la limpieza no perdió registros

Suscripción: validación superada
Telco: validación superada
(2800, 10) (4250, 21)


In [48]:
# Los datos originales no se modificaron
assert sus_raw['churn'].isin(['Yes', 'No']).all() and telco_raw['churn'].dtype == bool
print('Los DataFrame originales (sus_raw, telco_raw) siguen intactos y data/raw/ no se tocó.')

estado += [perfil_calidad(sus, 'suscripción', '2 tras limpieza (L2)'),
           perfil_calidad(telco, 'telco', '2 tras limpieza (L2)')]
pd.DataFrame(estado)

Los DataFrame originales (sus_raw, telco_raw) siguen intactos y data/raw/ no se tocó.


,dataset,etapa,filas,columnas,nulos,duplicados,columnas_texto_o_categoria,memoria_kb
0,suscripción,1 inicial,2800,10,0,0,3,278.0
1,telco,1 inicial,4250,21,0,0,3,841.6
2,suscripción,2 tras limpieza (L2),2800,10,0,0,1,73.1
3,telco,2 tras limpieza (L2),4250,21,0,0,3,536.2


In [49]:
display(Markdown(
    "**Resultado de la Lección 2:** ambos datasets quedan **limpios y validados**. No se eliminaron registros ni se imputaron valores, porque no había duplicados ni nulos. "
    "Lo que sí se corrigió fue el formato: fechas reales, `churn` en 0/1 en las dos tablas, categorías ordenadas, `area_code` reducido al número y enteros del tamaño justo."
))

**Resultado de la Lección 2:** ambos datasets quedan **limpios y validados**. No se eliminaron registros ni se imputaron valores, porque no había duplicados ni nulos. Lo que sí se corrigió fue el formato: fechas reales, `churn` en 0/1 en las dos tablas, categorías ordenadas, `area_code` reducido al número y enteros del tamaño justo.

---
# Lección 3 · Filtrado, ordenamiento y agrupación

Cada pregunta se traduce a "*métrica* por *grupo*". Aquí la métrica principal es la **tasa de churn** (porcentaje de clientes que cancelan; como `churn` es 0/1, el promedio es la tasa) y los grupos son los patrones de uso, soporte y pagos de la pregunta analítica.

## 3.1 Filtrado

In [50]:
def tasa(df):
    # porcentaje de clientes que cancelan; es el promedio de churn (0/1)
    return round(df['churn'].mean() * 100, 1)

print('Tasa de churn global  ·  suscripción:', tasa(sus), '%  ·  telco:', tasa(telco), '%')

Tasa de churn global  ·  suscripción: 57.3 %  ·  telco: 14.1 %


In [51]:
# Máscara booleana con varias condiciones (suscripción)
en_riesgo = sus[(sus['payment_failures'] >= 2) & (sus['last_login_days_ago'] >= 30)]

# isin(): clientes de los planes de menor costo
planes_basicos = sus[sus['plan_type'].isin(['Basic', 'Standard'])]

# between(): uso semanal bajo (de 0 a 5 horas)
poco_uso = sus[sus['avg_weekly_usage_hours'].between(0, 5)]

# query(): mucho soporte y poco uso
soporte_y_poco_uso = sus.query('support_tickets >= 5 and avg_weekly_usage_hours <= 5')

pd.DataFrame({
    'filtro': ['2+ fallos de pago y 30+ días sin entrar', 'planes Basic o Standard', 'uso de 0 a 5 h/semana', '5+ tickets y uso de 5 h o menos'],
    'clientes': [len(en_riesgo), len(planes_basicos), len(poco_uso), len(soporte_y_poco_uso)],
    'tasa_churn_%': [tasa(en_riesgo), tasa(planes_basicos), tasa(poco_uso), tasa(soporte_y_poco_uso)],
})

,filtro,clientes,tasa_churn_%
0,2+ fallos de pago y 30+ días sin entrar,945,73.9
1,planes Basic o Standard,1856,57.0
2,uso de 0 a 5 h/semana,523,76.3
3,5+ tickets y uso de 5 h o menos,226,84.5


In [52]:
# Mismos tipos de filtro en Telco
muchas_llamadas = telco[telco['number_customer_service_calls'] >= 4]
con_plan_intl = telco.query('international_plan')
uso_diurno_alto = telco[telco['total_day_minutes'].between(250, 400)]
intl_y_llamadas = telco.query('international_plan and number_customer_service_calls >= 4')
estados_top = telco[telco['state'].isin(['TX', 'NJ', 'MD'])]

pd.DataFrame({
    'filtro': ['4+ llamadas a servicio al cliente', 'plan internacional', 'más de 250 min diurnos', 'plan internacional y 4+ llamadas', 'estados TX, NJ o MD'],
    'clientes': [len(muchas_llamadas), len(con_plan_intl), len(uso_diurno_alto), len(intl_y_llamadas), len(estados_top)],
    'tasa_churn_%': [tasa(muchas_llamadas), tasa(con_plan_intl), tasa(uso_diurno_alto), tasa(intl_y_llamadas), tasa(estados_top)],
})

,filtro,clientes,tasa_churn_%
0,4+ llamadas a servicio al cliente,335,50.7
1,plan internacional,396,42.2
2,más de 250 min diurnos,423,48.9
3,plan internacional y 4+ llamadas,34,76.5
4,"estados TX, NJ o MD",280,22.9


In [53]:
g_sus, g_tel = tasa(sus), tasa(telco)
display(Markdown(
    f"**Interpretación:** la tasa global de cancelación es {g_sus} % en suscripción y {g_tel} % en Telco. "
    f"Los clientes con 2 o más fallos de pago y 30 o más días sin entrar cancelan en {tasa(en_riesgo)} % de los casos, y los clientes de Telco con 4 o más llamadas a servicio al cliente, en {tasa(muchas_llamadas)} %. "
    f"Los filtros ya muestran que los comportamientos de pago, ausencia y soporte se concentran donde hay más cancelaciones."
))

**Interpretación:** la tasa global de cancelación es 57.3 % en suscripción y 14.1 % en Telco. Los clientes con 2 o más fallos de pago y 30 o más días sin entrar cancelan en 73.9 % de los casos, y los clientes de Telco con 4 o más llamadas a servicio al cliente, en 50.7 %. Los filtros ya muestran que los comportamientos de pago, ausencia y soporte se concentran donde hay más cancelaciones.

## 3.2 Ordenamiento y selección de registros relevantes

In [54]:
# Los clientes de Telco con más llamadas a servicio al cliente (desempate: más minutos diurnos)
top_soporte = (telco.sort_values(['number_customer_service_calls', 'total_day_minutes'], ascending=False)
                    [['customer_id', 'number_customer_service_calls', 'total_day_minutes', 'international_plan', 'churn']]
                    .head(8))
top_soporte

,customer_id,number_customer_service_calls,total_day_minutes,international_plan,churn
3881,3e3023ee-affa-4d18-863d-ee41979b4e30,9,237.5,True,1
2693,91298599-8a25-4d55-a732-a63a8ca1ec6b,9,228.6,False,1
3655,9a29ef5f-6a5b-488f-93c5-9e0e626494b4,8,210.0,False,0
1597,fb3a4423-393f-4586-b2bc-4bd9090ebd40,8,150.6,False,1
1917,5711f993-6653-4f3c-9715-c6f6e634fb8c,7,304.2,False,1
4061,694cac27-60b5-4da2-94ac-fcf930506c0e,7,253.7,False,0
2843,a1cc0f65-601f-46df-ae56-391e32369bc3,7,232.2,True,1
386,ca659577-8979-460e-999a-59879371a646,7,207.2,False,0


In [55]:
# Suscripciones con más fallos de pago y más tiempo sin entrar: nlargest
sus.nlargest(8, ['payment_failures', 'last_login_days_ago'])[
    ['user_id', 'plan_type', 'payment_failures', 'last_login_days_ago', 'support_tickets', 'churn']]

,user_id,plan_type,payment_failures,last_login_days_ago,support_tickets,churn
1282,1283,Basic,5,60,3,1
1902,1903,Basic,5,60,0,1
2388,2389,Standard,5,60,0,1
2459,2460,Basic,5,60,7,1
2628,2629,Premium,5,60,3,1
8,9,Basic,5,59,4,1
26,27,Premium,5,59,2,0
578,579,Basic,5,59,7,1


In [56]:
top8 = top_soporte['churn'].sum()
display(Markdown(f"**Interpretación:** {top8} de los 8 clientes de Telco con más llamadas a soporte cancelaron. Ordenar y seleccionar los extremos permite ver rápido qué perfil acumula los problemas."))

**Interpretación:** 5 de los 8 clientes de Telco con más llamadas a soporte cancelaron. Ordenar y seleccionar los extremos permite ver rápido qué perfil acumula los problemas.

## 3.3 Categorizaciones con `pd.cut`

Los valores sueltos (por ejemplo, 3, 4 o 5 tickets) son difíciles de interpretar. Se agrupan en categorías con **cortes definidos a partir de lo observado en el diagnóstico**. Las funciones se reutilizan en la Lección 4 para construir los datasets finales.

In [57]:
def categorizar_suscripcion(df):
    df = df.copy()
    df['usage_group'] = pd.cut(df['avg_weekly_usage_hours'], bins=[0, 5, 15, 25],
                               labels=['bajo (<=5 h)', 'medio (5-15 h)', 'alto (>15 h)'])
    df['tickets_group'] = pd.cut(df['support_tickets'], bins=[-1, 2, 5, 8],
                                 labels=['0-2 tickets', '3-5 tickets', '6-8 tickets'])
    df['payment_group'] = pd.cut(df['payment_failures'], bins=[-1, 1, 5],
                                 labels=['0-1 fallos', '2-5 fallos'])
    df['recency_group'] = pd.cut(df['last_login_days_ago'], bins=[-1, 14, 29, 60],
                                 labels=['0-14 días', '15-29 días', '30-60 días'])
    # banderas de riesgo (1 = el cliente cumple la condición de riesgo)
    df['flag_low_usage'] = (df['avg_weekly_usage_hours'] <= 5).astype('int8')
    df['flag_many_tickets'] = (df['support_tickets'] >= 5).astype('int8')
    df['flag_payment_failures'] = (df['payment_failures'] >= 2).astype('int8')
    df['flag_inactive'] = (df['last_login_days_ago'] >= 30).astype('int8')
    flags = ['flag_low_usage', 'flag_many_tickets', 'flag_payment_failures', 'flag_inactive']
    df['risk_flags'] = df[flags].sum(axis=1).astype('int8')
    df['risk_level'] = pd.cut(df['risk_flags'], bins=[-1, 1, 2, 4], labels=['bajo (0-1)', 'medio (2)', 'alto (3-4)'])
    return df

def categorizar_telco(df):
    df = df.copy()
    df['service_calls_group'] = pd.cut(df['number_customer_service_calls'], bins=[-1, 1, 3, 20],
                                       labels=['0-1 llamadas', '2-3 llamadas', '4+ llamadas'])
    df['day_usage_group'] = pd.cut(df['total_day_minutes'], bins=[-1, 150, 250, np.inf],
                                   labels=['bajo (<=150 min)', 'medio (150-250 min)', 'alto (>250 min)'])
    df['flag_intl_plan'] = df['international_plan'].astype('int8')
    df['flag_many_calls'] = (df['number_customer_service_calls'] >= 4).astype('int8')
    df['flag_high_day_usage'] = (df['total_day_minutes'] > 250).astype('int8')
    flags = ['flag_intl_plan', 'flag_many_calls', 'flag_high_day_usage']
    df['risk_flags'] = df[flags].sum(axis=1).astype('int8')
    df['risk_level'] = pd.cut(df['risk_flags'], bins=[-1, 0, 1, 3], labels=['bajo (0)', 'medio (1)', 'alto (2-3)'])
    return df

sus_an = categorizar_suscripcion(sus)
telco_an = categorizar_telco(telco)

# Ningún cliente debe quedar sin categoría
cols_cat = ['usage_group', 'tickets_group', 'payment_group', 'recency_group', 'risk_level']
assert sus_an[cols_cat].notna().all().all(), 'hay clientes sin categoría (suscripción)'
assert telco_an[['service_calls_group', 'day_usage_group', 'risk_level']].notna().all().all(), 'hay clientes sin categoría (telco)'
print(sus_an['risk_level'].value_counts().sort_index().to_dict())
print(telco_an['risk_level'].value_counts().sort_index().to_dict())

{'bajo (0-1)': 1083, 'medio (2)': 1102, 'alto (3-4)': 615}
{'bajo (0)': 3216, 'medio (1)': 918, 'alto (2-3)': 116}


## 3.4 Agrupaciones con `groupby` y `agg`

Primero, un resumen general por plan (suscripción) y por tipo de plan internacional (Telco).

In [58]:
resumen_plan = (sus_an.groupby('plan_type', observed=True)
    .agg(clientes=('user_id', 'count'),
         churn_pct=('churn', lambda s: round(s.mean() * 100, 1)),
         uso_medio_h=('avg_weekly_usage_hours', 'mean'),
         tickets_medios=('support_tickets', 'mean'),
         fallos_pago_medios=('payment_failures', 'mean'))
    .round(2))
resumen_plan

,clientes,churn_pct,uso_medio_h,tickets_medios,fallos_pago_medios
plan_type,,,,,
Basic,923,57.9,13.16,3.81,2.51
Standard,933,56.1,12.90,3.93,2.52
Premium,944,58.1,12.63,3.92,2.44


In [59]:
resumen_intl = (telco_an.groupby('international_plan')
    .agg(clientes=('customer_id', 'count'),
         churn_pct=('churn', lambda s: round(s.mean() * 100, 1)),
         llamadas_soporte_medias=('number_customer_service_calls', 'mean'),
         minutos_diurnos_medios=('total_day_minutes', 'mean'))
    .round(2))
resumen_intl

,clientes,churn_pct,llamadas_soporte_medias,minutos_diurnos_medios
international_plan,,,,
False,3854,11.2,1.56,179.57
True,396,42.2,1.52,187.01


In [60]:
dif_planes = resumen_plan['churn_pct'].max() - resumen_plan['churn_pct'].min()
display(Markdown(
    f"**Interpretación:** el plan contratado casi no cambia la cancelación (la diferencia entre el mayor y el menor es de {dif_planes:.1f} puntos): el precio no explica el churn en esta base. "
    f"En Telco, en cambio, el plan internacional separa dos mundos: {resumen_intl.loc[True, 'churn_pct']} % de churn con plan internacional frente a {resumen_intl.loc[False, 'churn_pct']} % sin él."
))

**Interpretación:** el plan contratado casi no cambia la cancelación (la diferencia entre el mayor y el menor es de 2.0 puntos): el precio no explica el churn en esta base. En Telco, en cambio, el plan internacional separa dos mundos: 42.2 % de churn con plan internacional frente a 11.2 % sin él.

## 3.5 Indicadores de churn ligados a la pregunta analítica

Se calculan **seis indicadores**, agrupados por los tres temas de la pregunta: *interacción y frecuencia de uso*, *soporte técnico* y *pagos*. Todos usan la misma función, que devuelve el número de clientes, el de cancelaciones y la tasa de churn de cada grupo.

In [61]:
def tabla_tasa(df, col, dataset, indicador):
    g = df.groupby(col, observed=True)['churn']
    t = pd.DataFrame({'clientes': g.size(), 'cancelaciones': g.sum(), 'tasa_churn_pct': (g.mean() * 100).round(1)}).reset_index()
    t = t.rename(columns={col: 'grupo'})
    t['grupo'] = t['grupo'].astype(str)
    t.insert(0, 'indicador', indicador)
    t.insert(0, 'dataset', dataset)
    return t

ind1 = tabla_tasa(sus_an, 'usage_group', 'suscripción', 'IND-1 uso semanal')
ind2 = tabla_tasa(sus_an, 'recency_group', 'suscripción', 'IND-2 días sin iniciar sesión')
ind3 = tabla_tasa(sus_an, 'tickets_group', 'suscripción', 'IND-3 tickets de soporte')
ind4 = tabla_tasa(sus_an, 'payment_group', 'suscripción', 'IND-4 fallos de pago')
ind5 = tabla_tasa(telco_an, 'service_calls_group', 'telco', 'IND-5 llamadas a servicio al cliente')
ind6 = tabla_tasa(telco_an, 'day_usage_group', 'telco', 'IND-6 uso diurno')
indicadores = pd.concat([ind1, ind2, ind3, ind4, ind5, ind6], ignore_index=True)
indicadores

,dataset,indicador,grupo,clientes,cancelaciones,tasa_churn_pct
0,suscripción,IND-1 uso semanal,bajo (<=5 h),523,399,76.3
1,suscripción,IND-1 uso semanal,medio (5-15 h),1100,575,52.3
2,suscripción,IND-1 uso semanal,alto (>15 h),1177,631,53.6
3,suscripción,IND-2 días sin iniciar sesión,0-14 días,709,297,41.9
4,suscripción,IND-2 días sin iniciar sesión,15-29 días,690,383,55.5
5,suscripción,IND-2 días sin iniciar sesión,30-60 días,1401,925,66.0
6,suscripción,IND-3 tickets de soporte,0-2 tickets,994,490,49.3
7,suscripción,IND-3 tickets de soporte,3-5 tickets,882,512,58.0
8,suscripción,IND-3 tickets de soporte,6-8 tickets,924,603,65.3
9,suscripción,IND-4 fallos de pago,0-1 fallos,938,374,39.9


In [62]:
def lectura(t, nombre, unidad=''):
    # frase con el grupo de mayor y menor churn de un indicador
    hi, lo = t.loc[t['tasa_churn_pct'].idxmax()], t.loc[t['tasa_churn_pct'].idxmin()]
    return (f"**{nombre}:** el grupo *{hi['grupo']}* tiene la mayor tasa de churn ({hi['tasa_churn_pct']} %) y *{lo['grupo']}* la menor ({lo['tasa_churn_pct']} %); "
            f"diferencia de {hi['tasa_churn_pct'] - lo['tasa_churn_pct']:.1f} puntos.")

display(Markdown("**Interpretación de los indicadores**\n\n" + "\n\n".join([
    lectura(ind1, 'IND-1 uso semanal'),
    lectura(ind2, 'IND-2 sin iniciar sesión'),
    lectura(ind3, 'IND-3 tickets de soporte'),
    lectura(ind4, 'IND-4 fallos de pago'),
    lectura(ind5, 'IND-5 llamadas a servicio al cliente'),
    lectura(ind6, 'IND-6 uso diurno'),
])))

**Interpretación de los indicadores**

**IND-1 uso semanal:** el grupo *bajo (<=5 h)* tiene la mayor tasa de churn (76.3 %) y *medio (5-15 h)* la menor (52.3 %); diferencia de 24.0 puntos.

**IND-2 sin iniciar sesión:** el grupo *30-60 días* tiene la mayor tasa de churn (66.0 %) y *0-14 días* la menor (41.9 %); diferencia de 24.1 puntos.

**IND-3 tickets de soporte:** el grupo *6-8 tickets* tiene la mayor tasa de churn (65.3 %) y *0-2 tickets* la menor (49.3 %); diferencia de 16.0 puntos.

**IND-4 fallos de pago:** el grupo *2-5 fallos* tiene la mayor tasa de churn (66.1 %) y *0-1 fallos* la menor (39.9 %); diferencia de 26.2 puntos.

**IND-5 llamadas a servicio al cliente:** el grupo *4+ llamadas* tiene la mayor tasa de churn (50.7 %) y *0-1 llamadas* la menor (10.9 %); diferencia de 39.8 puntos.

**IND-6 uso diurno:** el grupo *alto (>250 min)* tiene la mayor tasa de churn (48.9 %) y *medio (150-250 min)* la menor (9.4 %); diferencia de 39.5 puntos.

### Qué dicen los indicadores sobre la pregunta analítica

- **Interacción y frecuencia de uso:** IND-1 e IND-2 describen si el cliente se está alejando de la plataforma. Un uso semanal muy bajo y muchos días sin iniciar sesión van de la mano con una mayor cancelación.
- **Soporte técnico:** IND-3 (tickets) e IND-5 (llamadas a servicio al cliente) miden la fricción con el servicio. En Telco hay un **umbral**: hasta 3 llamadas la tasa es parecida a la del resto, y desde la cuarta se dispara.
- **Pagos:** IND-4 muestra que los fallos de pago son una señal fuerte en suscripción.
- **Uso en Telco:** IND-6 muestra el patrón contrario al de suscripción: el uso diurno **muy alto** (más de 250 minutos) se asocia con más cancelación, mientras que el uso medio es el que menos cancela. Es una pista de que el costo de la factura puede pesar en este negocio.
- Ninguno de estos indicadores usa una ventana de 30 días: el dataset no la trae. Lo que se mide es el comportamiento **acumulado** hasta el momento del registro.

## 3.6 Tablas dinámicas y combinación de señales

In [63]:
# Tabla dinámica: churn (%) según fallos de pago y tickets de soporte
pivot_sus = (sus_an.pivot_table(index='payment_group', columns='tickets_group', values='churn',
                                aggfunc='mean', observed=True) * 100).round(1)
pivot_sus

tickets_group,0-2 tickets,3-5 tickets,6-8 tickets
payment_group,,,
0-1 fallos,34.2,39.7,45.8
2-5 fallos,56.9,66.7,75.7


In [64]:
# Tabla dinámica en Telco: plan internacional y llamadas a servicio al cliente
pivot_telco = (telco_an.pivot_table(index='international_plan', columns='service_calls_group', values='churn',
                                    aggfunc='mean', observed=True) * 100).round(1)
pivot_telco

service_calls_group,0-1 llamadas,2-3 llamadas,4+ llamadas
international_plan,,,
False,8.0,8.2,47.8
True,38.0,40.6,76.5


In [65]:
def tabla_riesgo(df, dataset):
    t = tabla_tasa(df, 'risk_flags', dataset, 'banderas de riesgo acumuladas')
    t['veces_la_tasa_global'] = (t['tasa_churn_pct'] / (df['churn'].mean() * 100)).round(2)
    return t

riesgo_sus = tabla_riesgo(sus_an, 'suscripción')
riesgo_telco = tabla_riesgo(telco_an, 'telco')
display(riesgo_sus)
display(riesgo_telco)

,dataset,indicador,grupo,clientes,cancelaciones,tasa_churn_pct,veces_la_tasa_global
0,suscripción,banderas de riesgo acumuladas,0,205,43,21.0,0.37
1,suscripción,banderas de riesgo acumuladas,1,878,361,41.1,0.72
2,suscripción,banderas de riesgo acumuladas,2,1102,683,62.0,1.08
3,suscripción,banderas de riesgo acumuladas,3,538,441,82.0,1.43
4,suscripción,banderas de riesgo acumuladas,4,77,77,100.0,1.74


,dataset,indicador,grupo,clientes,cancelaciones,tasa_churn_pct,veces_la_tasa_global
0,telco,banderas de riesgo acumuladas,0,3216,128,4.0,0.28
1,telco,banderas de riesgo acumuladas,1,918,399,43.5,3.09
2,telco,banderas de riesgo acumuladas,2,112,68,60.7,4.31
3,telco,banderas de riesgo acumuladas,3,4,3,75.0,5.33


In [66]:
r0, r4 = riesgo_sus.iloc[0], riesgo_sus.iloc[-1]
rt0, rtm = riesgo_telco.iloc[0], riesgo_telco.iloc[-1]
display(Markdown(
    f"**Interpretación:** las señales **se acumulan**. En suscripción, los clientes sin ninguna bandera de riesgo cancelan en {r0['tasa_churn_pct']} % de los casos y los que cumplen {int(r4['grupo'])} cancelan en {r4['tasa_churn_pct']} % "
    f"({r4['veces_la_tasa_global']} veces la tasa global). En Telco, con {rt0['grupo']} banderas la tasa es {rt0['tasa_churn_pct']} % y con {rtm['grupo']} llega a {rtm['tasa_churn_pct']} % ({rtm['veces_la_tasa_global']} veces la global). "
    f"Esto sugiere que un modelo de riesgo debería combinar variables de uso, soporte y pagos y no depender de una sola."
))

**Interpretación:** las señales **se acumulan**. En suscripción, los clientes sin ninguna bandera de riesgo cancelan en 21.0 % de los casos y los que cumplen 4 cancelan en 100.0 % (1.74 veces la tasa global). En Telco, con 0 banderas la tasa es 4.0 % y con 3 llega a 75.0 % (5.33 veces la global). Esto sugiere que un modelo de riesgo debería combinar variables de uso, soporte y pagos y no depender de una sola.